In [9]:
!ls ../../../../assets/gender/user_embeddings/embeddings.parquet

../../../../assets/gender/user_embeddings/embeddings.parquet


In [1]:
import pandas as pd 
import numpy as np

from typing import List, Tuple, Set, Dict

In [3]:
!ls ../data/

gender_train.csv       test_ids.csv	  tr_type_mapping.json
mcc_code_mapping.json  test_trx.parquet   tr_types.csv
mcc_dict.npy	       train_trx.parquet  tr_type_translated.csv
mcc_translated.csv     transactions.csv   type_dict.npy
term_id_mapping.json   tr_mcc_codes.csv


In [4]:
df = pd.read_csv('../data/transactions.csv')
df.head()

,customer_id,tr_datetime,mcc_code,tr_type,amount,term_id
0,39026145,0 10:23:26,4814,1030,-2245.92,NaN
1,39026145,1 10:19:29,6011,7010,56147.89,NaN
2,39026145,1 10:20:56,4829,2330,-56147.89,NaN
3,39026145,1 10:39:54,5499,1010,-1392.47,NaN
4,39026145,2 15:33:42,5499,1010,-920.83,NaN


In [5]:
# average number of rows per customer_id
df.groupby('customer_id').size().mean()

456.62306666666666

In [6]:
# average number of rows per customer_id
df.groupby('customer_id').size().max()

88781

In [7]:
df.groupby('customer_id').size().min()

1

In [9]:
df.groupby('customer_id').size().median()

324.5

In [10]:
df.shape

(6849346, 6)

In [3]:
train_set_path: str = '../data/train_trx.parquet'
test_set_path: str = '../data/test_trx.parquet'

df_train = pd.read_parquet(train_set_path)
df_test = pd.read_parquet(test_set_path)

df_train['customer_id'] = df_train['customer_id'].astype('int64')
df_test['customer_id'] = df_test['customer_id'].astype('int64')

In [4]:
df_test.head()

,customer_id,event_time,amount,mcc_code,tr_type,term_id,trx_count,target
0,24227698,"[2.7778125, 7.502939814814815, 17.000011574074...","[-10.424948618947251, -11.811220643640558, -11...","[2, 2, 2, 2, 2, 2, 4, 2, 2, 2, 2, 2, 2, 2, 2, ...","[3, 3, 3, 3, 8, 3, 4, 8, 3, 3, 8, 3, 3, 8, 8, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",235,1
1,29097966,"[0.0, 0.3258449074074074, 2.0, 2.0, 3.42097222...","[-8.79882449483086, -7.024613442631785, -9.204...","[8, 4, 8, 5, 4, 2, 8, 9, 2, 4, 4, 8, 26, 2, 5,...","[5, 4, 5, 5, 4, 3, 5, 5, 3, 4, 4, 5, 2, 3, 2, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",1255,1
2,46431440,"[12.414259259259259, 14.528055555555556, 14.57...","[10.974982295004144, -12.227733055882192, -8.4...","[2, 2, 2, 2, 4, 2, 2, 4, 2, 4, 2, 2, 4, 2, 2, ...","[8, 3, 18, 3, 4, 3, 8, 4, 3, 4, 18, 3, 4, 3, 1...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",144,0
3,48119333,"[1.0, 1.241886574074074, 1.5754398148148148, 3...","[-8.969759419013014, 11.118080809668006, 11.62...","[49, 3, 3, 2, 8, 3, 4, 8, 8, 76, 5, 4, 4, 4, 5...","[10, 6, 9, 16, 5, 6, 4, 5, 5, 5, 5, 4, 4, 4, 5...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",632,1
4,50796712,"[1.1659143518518518, 1.4113194444444446, 1.418...","[-10.722058283317635, -7.773670190197818, -9.3...","[8, 5, 5, 5, 2, 5, 9, 5, 4, 5, 5, 4, 5, 5, 5, ...","[5, 2, 5, 2, 3, 5, 5, 5, 4, 5, 5, 4, 5, 5, 5, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...",455,1


In [5]:
mcc_codes: Set[int] = set()
tr_types: Set[str] = set()

for df in [df_train, df_test]:
    for _, row in df.iterrows():
        mcc_codes.update(row['mcc_code'])
        tr_types.update(row['tr_type'])

In [6]:
mcc_codes: List[int] = sorted(mcc_codes)
tr_types: List[str] = sorted(tr_types)

In [7]:
import numpy as np  
from numpy.typing import NDArray

def extract_event_time_features(data: List[float]) -> Tuple[np.float64, np.float64]:
    time_diffs = np.diff(data)
    if len(time_diffs) == 0:
        return 0.0, 0.0
    
    return np.mean(time_diffs), np.std(time_diffs)

def extract_amount_features(data: List[float]) -> Tuple[np.float64, np.float64, np.float64, np.float64]:
    if len(data) == 0:
        return 0.0, 0.0, 0.0, 0.0
    
    return np.min(data), np.max(data), np.mean(data), np.std(data)

def extract_mcc_features(data: List[int]) -> NDArray:
    mcc_vector = np.zeros(len(mcc_codes))
    for mcc in data:
        if mcc in mcc_codes:
            idx = mcc_codes.index(mcc)
            mcc_vector[idx] += 1
    return mcc_vector

def extract_transaction_type_features(data: List[int]) -> NDArray:
    tr_type_vector = np.zeros(len(tr_types))
    for tr_type in data:
        if tr_type in tr_types:
            idx = tr_types.index(tr_type)
            tr_type_vector[idx] += 1
    return tr_type_vector

def extract_user_features(df: pd.DataFrame) -> Tuple[NDArray, NDArray]:
    X = []
    y = []

    for _, row in df.iterrows():
        # if target is nan, skip the row
        if pd.isna(row['target']):
            continue

        event_time_features = extract_event_time_features(row['event_time'])
        amount_features = extract_amount_features(row['amount'])
        mcc_features = extract_mcc_features(row['mcc_code'])
        tr_type_features = extract_transaction_type_features(row['tr_type'])
        num_transactions = row['trx_count']

        user_features = np.concatenate([event_time_features, amount_features, mcc_features, tr_type_features, [num_transactions]])
        X.append(user_features)
        y.append(row['target'])

    return np.array(X), np.array(y)

In [8]:
X_train, y_train = extract_user_features(df_train)
X_test, y_test = extract_user_features(df_test)

In [9]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((7560, 268), (7560,), (840, 268), (840,))

In [10]:
import lightgbm as lgbm

model = lgbm.LGBMClassifier(
    n_estimators=100, boosting_type='gbdt', objective='binary', metric='auc', subsample=0.5,
    subsample_freq=1, learning_rate=0.02, feature_fraction=0.75, max_depth=6, lambda_l1=1,
    lambda_l2=1, min_data_in_leaf=50, random_state=42, n_jobs=8, reg_alpha=0, reg_lambda=0,
    colsample_bytree=0, min_child_samples=0, num_iterations = 1000
)

In [11]:
model.fit(X_train, y_train)

/home/elves/third/llm4es/src/ptls-experiments/venv/lib/python3.10/site-packages/lightgbm/engine.py:177: UserWarning: Found `num_iterations` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")


[LightGBM] [Warning] lambda_l1 is set=1, reg_alpha=0 will be ignored. Current value: lambda_l1=1
[LightGBM] [Warning] feature_fraction is set=0.75, colsample_bytree=0 will be ignored. Current value: feature_fraction=0.75
[LightGBM] [Warning] lambda_l2 is set=1, reg_lambda=0 will be ignored. Current value: lambda_l2=1
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=0 will be ignored. Current value: min_data_in_leaf=50


LGBMClassifier(colsample_bytree=0, feature_fraction=0.75, lambda_l1=1,
               lambda_l2=1, learning_rate=0.02, max_depth=6, metric='auc',
               min_child_samples=0, min_data_in_leaf=50, n_jobs=8,
               num_iterations=1000, objective='binary', random_state=42,
               reg_alpha=0, reg_lambda=0, subsample=0.5, subsample_freq=1)

In [12]:
from sklearn.metrics import roc_auc_score

# check model performance on test set
y_pred = model.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_pred)
print(f"AUC score on test set: {auc_score:.4f}")

# acuracy, precision, recall, f1-score, confusion matrix, classification report, etc.

acuracy = (y_pred.round() == y_test).mean()
print(f"Acuracy on test set: {acuracy:.4f}")

AUC score on test set: 0.8854
Acuracy on test set: 0.8036


---

Now let's use the user embeddings to train a downstream model for gender

In [13]:
user_embeddings_path: str = '../../../../assets/gender/user_embeddings/embeddings.parquet'
df_user_embeddings = pd.read_parquet(user_embeddings_path)

df_user_embeddings.head()

,customer_id,emb_0001,emb_0002,emb_0003,emb_0004,emb_0005,emb_0006,emb_0007,emb_0008,emb_0009,...,emb_4087,emb_4088,emb_4089,emb_4090,emb_4091,emb_4092,emb_4093,emb_4094,emb_4095,emb_4096
0,10058778,-0.111328,0.007324,-0.008362,0.030518,0.108887,0.011108,-0.070801,0.018311,-0.037109,...,-0.029175,-0.086914,0.069824,-0.085449,-0.024902,0.041748,0.161133,-0.024292,0.000916,-0.017090
1,10230827,-0.110840,-0.041992,-0.013977,-0.016602,0.121582,-0.008972,-0.057617,0.018799,-0.043701,...,0.000576,-0.099609,0.017090,-0.137695,-0.055664,-0.003342,0.184570,-0.042480,-0.008667,-0.017822
2,11681378,-0.098633,-0.016479,-0.039795,-0.039551,0.115723,-0.025269,-0.055176,0.022705,-0.012878,...,-0.047852,-0.101562,0.018921,-0.144531,-0.055664,0.000881,0.182617,-0.047607,-0.009094,-0.017212
3,14123285,-0.107422,0.008850,-0.011169,-0.004730,0.133789,0.012573,-0.072754,0.020142,-0.014526,...,-0.018433,-0.085449,0.052002,-0.098633,-0.032715,0.039551,0.175781,-0.010559,-0.020996,-0.022827
4,16536678,-0.105469,0.012939,-0.013062,0.002777,0.115723,-0.001060,-0.055664,0.057617,-0.067383,...,-0.031494,-0.091309,0.063477,-0.042969,0.024536,0.083008,0.155273,0.080078,-0.067871,-0.034912


In [16]:
# first, we copy the df_train with only the customer_id and target columns
df_train_copy = df_train[['customer_id', 'target']].copy()
# then, we merge the df_train_copy with the df_user_embeddings on customer_id
df_train_emb = df_train_copy.merge(df_user_embeddings, on='customer_id', how='inner')

df_test_copy = df_test[['customer_id', 'target']].copy()
df_test_emb = df_test_copy.merge(df_user_embeddings, on='customer_id', how='inner')

In [20]:
embedding_cols = [col for col in df_user_embeddings.columns if col.startswith('emb_')]

In [26]:


def extract_user_embedding_features(df: pd.DataFrame, emb_cols: List[str], target_col: str = 'target') -> Tuple[NDArray, NDArray]:
    # filter rows with target is nan
    df = df[~df[target_col].isna()]
    
    X = df[emb_cols].values
    y = df[target_col].values 

    return np.array(X), np.array(y)

X_train_emb, y_train_emb = extract_user_embedding_features(df_train_emb, embedding_cols)
X_test_emb, y_test_emb = extract_user_embedding_features(df_test_emb, embedding_cols)

In [27]:
model2 = lgbm.LGBMClassifier(
    n_estimators=100, boosting_type='gbdt', objective='binary', metric='auc', subsample=0.5,
    subsample_freq=1, learning_rate=0.02, feature_fraction=0.75, max_depth=6, lambda_l1=1,
    lambda_l2=1, min_data_in_leaf=50, random_state=42, n_jobs=8, reg_alpha=0, reg_lambda=0,
    colsample_bytree=0, min_child_samples=0, num_iterations = 1000
)

In [28]:
model2.fit(X_train_emb, y_train_emb)

/home/elves/third/llm4es/src/ptls-experiments/venv/lib/python3.10/site-packages/lightgbm/engine.py:177: UserWarning: Found `num_iterations` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")


[LightGBM] [Warning] lambda_l1 is set=1, reg_alpha=0 will be ignored. Current value: lambda_l1=1
[LightGBM] [Warning] feature_fraction is set=0.75, colsample_bytree=0 will be ignored. Current value: feature_fraction=0.75
[LightGBM] [Warning] lambda_l2 is set=1, reg_lambda=0 will be ignored. Current value: lambda_l2=1
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=0 will be ignored. Current value: min_data_in_leaf=50


LGBMClassifier(colsample_bytree=0, feature_fraction=0.75, lambda_l1=1,
               lambda_l2=1, learning_rate=0.02, max_depth=6, metric='auc',
               min_child_samples=0, min_data_in_leaf=50, n_jobs=8,
               num_iterations=1000, objective='binary', random_state=42,
               reg_alpha=0, reg_lambda=0, subsample=0.5, subsample_freq=1)

In [ ]:
# check model performance on test set
y_pred_emb = model2.predict_proba(X_test_emb)[:, 1]
auc_score_emb = roc_auc_score(y_test_emb, y_pred_emb)
print(f"AUC score on test set with embeddings: {auc_score_emb:.4f}")


# acuracy, precision, recall, f1-score, confusion matrix, classification report, etc.
acuracy = (y_pred_emb.round() == y_test_emb).mean()
print(f"Acuracy on test set: {acuracy:.4f}")

AUC score on test set with embeddings: 0.8151
Acuracy on test set: 0.7393
